# Generate Competition Submission File

Creates the final submission CSV for the competition test data using the trained ensemble model.

## Configuration

In [ ]:
# File paths
TEST_FILE = "../output/test_with_features_encoded.csv"  # Test data with features
TRAIN_FILE = "../output/train_split.csv"  # For feature column names

# Model files
RF_MODELS = "../output/rf_models.pkl"
XGB_MODELS = "../output/xgb_models.pkl"
ENSEMBLE_WEIGHTS = "../output/ensemble_weights.pkl"

# Output
SUBMISSION_FILE = "../output/submission.csv"
PROBA_FILE = "../output/submission_probabilities.csv"

# Prediction threshold
THRESHOLD = 0.5

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
import warnings
warnings.filterwarnings("ignore")

## Load Test Data

In [ ]:
print("Loading test data...")
test_df = pd.read_csv(TEST_FILE)
print(f"✓ Test samples: {len(test_df):,}")

## Load Feature Columns

In [ ]:
# Get feature column names from training data
train_sample = pd.read_csv(TRAIN_FILE, nrows=1)
label_cols = ["S-glutathionylation", "S-nitrosylation", "S-palmitoylation"]
feature_cols = [col for col in train_sample.columns if col not in ["ID", "Sequence"] + label_cols]

print(f"✓ Feature columns: {len(feature_cols)}")

## Prepare Test Features

In [ ]:
# Ensure test has same features as training
available = [col for col in feature_cols if col in test_df.columns]
missing = set(feature_cols) - set(available)

if missing:
    print(f"⚠️ Warning: {len(missing)} features missing, filling with zeros")
    for col in missing:
        test_df[col] = 0

# Get features in correct order
X_test_df = test_df[feature_cols].copy()

# Convert bool to int
for col in X_test_df.columns:
    if X_test_df[col].dtype == "bool":
        X_test_df[col] = X_test_df[col].astype(int)

# Remove non-numeric
non_numeric = X_test_df.select_dtypes(include=["object"]).columns
if len(non_numeric) > 0:
    X_test_df = X_test_df.drop(columns=non_numeric)

X_test = X_test_df.values.astype(float)
print(f"✓ Test features prepared: {X_test.shape}")

## Make Predictions

In [ ]:
predictions = {}

# RF predictions
if os.path.exists(RF_MODELS):
    with open(RF_MODELS, "rb") as f:
        rf_models = pickle.load(f)
    rf_preds = np.column_stack([rf_models[label].predict_proba(X_test)[:, 1] for label in label_cols])
    predictions["rf"] = rf_preds
    print("✓ RF predictions")

# XGBoost predictions
if os.path.exists(XGB_MODELS):
    with open(XGB_MODELS, "rb") as f:
        xgb_models = pickle.load(f)
    xgb_preds = np.column_stack([xgb_models[label].predict_proba(X_test)[:, 1] for label in label_cols])
    predictions["xgb"] = xgb_preds
    print("✓ XGBoost predictions")

print(f"\nTotal models: {len(predictions)}")

## Create Ensemble

In [ ]:
# Load ensemble weights
if os.path.exists(ENSEMBLE_WEIGHTS):
    with open(ENSEMBLE_WEIGHTS, "rb") as f:
        weights = pickle.load(f)
    print("✓ Loaded ensemble weights:")
    for model, weight in weights.items():
        print(f"  {model}: {weight:.2f}")
    
    # Weighted ensemble
    weighted_sum = sum(weights.get(m, 0) * predictions[m] for m in predictions.keys())
    total_weight = sum(weights.get(m, 0) for m in predictions.keys())
    final_proba = weighted_sum / total_weight
    
else:
    print("⚠️ No ensemble weights, using simple average")
    final_proba = np.mean(list(predictions.values()), axis=0)

print(f"\nFinal predictions shape: {final_proba.shape}")

## Create Submission File

In [ ]:
# Binary predictions
final_binary = (final_proba > THRESHOLD).astype(int)

# Create submission DataFrame
submission = pd.DataFrame({
    "ID": test_df["ID"],
    "Sequence": test_df["Sequence"],
    "S-glutathionylation": final_binary[:, 0],
    "S-nitrosylation": final_binary[:, 1],
    "S-palmitoylation": final_binary[:, 2]
})

# Save submission
submission.to_csv(SUBMISSION_FILE, index=False)
print(f"✓ Saved {SUBMISSION_FILE}")

# Save probabilities for reference
proba_df = pd.DataFrame({
    "ID": test_df["ID"],
    "S-glutathionylation": final_proba[:, 0],
    "S-nitrosylation": final_proba[:, 1],
    "S-palmitoylation": final_proba[:, 2]
})
proba_df.to_csv(PROBA_FILE, index=False)
print(f"✓ Saved {PROBA_FILE}")

## Submission Summary

In [ ]:
print("\n" + "="*80)
print("SUBMISSION SUMMARY")
print("="*80)
print(f"\nTotal samples: {len(submission):,}")
print(f"\nPrediction distribution:")

for label in label_cols:
    pos = submission[label].sum()
    pct = pos / len(submission) * 100
    print(f"  {label}: {pos:,} positive ({pct:.2f}%)")

print("\n" + "="*80)
print(f"✓ SUBMISSION READY: {SUBMISSION_FILE}")
print("="*80)